# VRP decay check: does the 200-day gate still tame the tail post-2019? (Colab, token-free)

Same VRP short-variance harvest as before, but run over THREE eras automatically — **full history, 2010+,
and 2019+** — side by side. The question: is the '200-day filter cuts the drawdown ~8x' result still alive
in the modern (post-volmageddon / post-COVID) regime, or did it decay? No edits — just run top-to-bottom.


## capture setup


In [ ]:
import builtins as _bi
REPORT_LINES=[]; _realprint=_bi.print
def print(*a, sep=' ', end='\n', **k):
    _realprint(*a, sep=sep, end=end, **k)
    try: REPORT_LINES.append(sep.join(str(x) for x in a))
    except Exception: pass
_realprint('capture ON')


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np
except Exception:
    _pip('pandas'); _pip('numpy'); import pandas as pd, numpy as np
def yf_close(sym, start='1990-01-01'):
    try:
        import yfinance as yf
    except Exception:
        _pip('yfinance'); import yfinance as yf
    df = yf.download(sym, start=start, progress=False, auto_adjust=True)
    if not len(df): return None
    c = df['Close']; c = c.iloc[:,0] if hasattr(c,'columns') else c
    return pd.Series(np.asarray(c).ravel(), index=df.index).dropna()
def stooq(sym):
    import urllib.request, io
    raw = urllib.request.urlopen(f'https://stooq.com/q/d/l/?s={sym}&i=d', timeout=30).read().decode()
    df = pd.read_csv(io.StringIO(raw), parse_dates=['Date']).set_index('Date').sort_index()
    return df['Close'].dropna()
def load(a,b):
    try:
        s = yf_close(a)
        if s is not None and len(s)>500: return s
    except Exception as e: print(a,'yf failed:',e)
    return stooq(b)

spx = load('^GSPC','^spx'); vix = load('^VIX','^vix')
df = pd.DataFrame({'spx':spx,'vix':vix}).dropna()
df['ret'] = df['spx'].pct_change()
# precompute on FULL continuous history, then slice by era (avoids edge effects)
H=21
df['rv_fwd'] = df['ret'].shift(-1).rolling(H).std().shift(-(H-1))*np.sqrt(252)*100
df['vrp']    = df['vix'] - df['rv_fwd']
df['carry']  = (df['vix']/100.0)**2/252.0
df['realized']= df['ret'].shift(-1)**2
df['pnl']    = (df['carry']-df['realized'])*1e4
df['sma200'] = df['spx'].rolling(200).mean(); df['bull']=df['spx']>df['sma200']
print('loaded', len(df), 'days', df.index[0].date(), '->', df.index[-1].date())


## Run the harvest over each era


In [ ]:
def cstats(pnl):
    pnl=pnl.dropna();
    if len(pnl)<50: return None
    eq=pnl.cumsum(); dd=eq-eq.cummax()
    sh=pnl.mean()/pnl.std()*np.sqrt(252) if pnl.std()>0 else float('nan')
    return dict(total=eq.iloc[-1], sharpe=sh, maxDD=dd.min(), worst=pnl.min(), up=100*(pnl>0).mean())

def analyze(sub, label):
    print('\n'+'='*58); print(f'ERA: {label}   ({sub.index[0].date()} -> {sub.index[-1].date()}, {len(sub)} days)'); print('='*58)
    v=sub['vrp'].dropna()
    print(f'  VRP: mean {v.mean():+.2f} pts | seller wins {100*(v>0).mean():.0f}% | worst {v.min():+.0f}')
    med=sub['vix'].median()
    variants=[('ALWAYS short', sub['pnl']),
              ('only VIX>median', sub.loc[sub['vix']>med,'pnl']),
              ('only BULL(>200d)', sub.loc[sub['bull'],'pnl']),
              ('BULL & VIX>med', sub.loc[sub['bull']&(sub['vix']>med),'pnl'])]
    for nm,p in variants:
        s=cstats(p)
        if s: print(f'  {nm:17s}: total {s["total"]:7.0f} | Sharpe {s["sharpe"]:5.2f} | maxDD {s["maxDD"]:7.0f} | worst {s["worst"]:6.0f} | up {s["up"]:3.0f}%')
    a=cstats(sub['pnl']); b=cstats(sub.loc[sub['bull'],'pnl'])
    if a and b:
        print(f'  >> 200-day gate effect: drawdown {a["maxDD"]:.0f} -> {b["maxDD"]:.0f} '
              f'({abs(b["maxDD"])/abs(a["maxDD"]):.0%} of the risk), income kept {b["total"]/a["total"]:.0%}')

for label, start in [('FULL history', None), ('2010+', '2010-01-01'), ('2019+ (modern)', '2019-01-01')]:
    sub = df if start is None else df[df.index>=start]
    analyze(sub, label)
print('\nDECAY READ: compare the 200-day-gate line across eras. If bull-only still cuts the drawdown hard')
print('and keeps most of the income in 2019+, the edge is LIVE. If the gate stops helping, it decayed.')


## ⬇️ EXPORT — run LAST


In [ ]:
from datetime import datetime
fname='vrp_decay_report.txt'
hdr=['='*60,'VRP DECAY CHECK (full / 2010+ / 2019+) — export',
     f'generated {datetime.now().strftime("%Y-%m-%d %H:%M")}','='*60,'']
open(fname,'w').write('\n'.join(hdr)+'\n'+'\n'.join(REPORT_LINES)+'\n')
_realprint('wrote',fname,'—',len(REPORT_LINES),'lines')
if not REPORT_LINES: _realprint('!! empty — run cells above first')
try:
    from google.colab import files; files.download(fname); _realprint('download started')
except Exception as e:
    _realprint('not in Colab / blocked:',e,'- grab it from the folder icon (left)')
